# Image Homographies: A Self-Learning Notebook for Computer Vision

This notebook is designed as a self-learning chapter for students studying **image homographies** in computer vision.

The structure follows a simple pattern:

1. Read the concept first.
2. Run the code example.
3. Complete the task or reflection question.
4. Compare the result with your intuition.

## Learning objectives

By the end of this notebook, you should be able to:

- Explain when an image homography is a valid model.
- Apply a homography to 2D points using homogeneous coordinates.
- Warp an image using a homography.
- Estimate a homography from point correspondences using the Direct Linear Transform, or DLT.
- Improve DLT using point normalization.
- Use RANSAC to estimate a homography when some correspondences are outliers.
- Understand how homographies support image rectification and panorama-style image alignment.

## Notebook rules

- All images are generated inside the notebook, so no external data is required.
- Every main section has a concept block before the code block.
- Mathematical expressions use Jupyter-stable inline math with `$...$` and display math with `$$...$$`.


## 0. Setup

This notebook uses `numpy`, `matplotlib`, and `opencv-python`.

OpenCV is used for image warping and feature matching. The homography estimation code is implemented from scratch so that you can see the math behind the algorithm.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

plt.rcParams["figure.figsize"] = (6, 6)
plt.rcParams["image.cmap"] = "gray"


def show_image(img, title=None, size=(6, 6)):
    """Display a grayscale or RGB image."""
    plt.figure(figsize=size)
    if img.ndim == 2:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()


def plot_points(points, labels=None, title=None, size=(6, 6), axis_equal=True):
    """Plot 2D points with optional labels."""
    points = np.asarray(points)
    plt.figure(figsize=size)
    plt.scatter(points[:, 0], points[:, 1])
    if labels is not None:
        for p, label in zip(points, labels):
            plt.text(p[0] + 2, p[1] + 2, str(label))
    if axis_equal:
        plt.axis("equal")
    plt.gca().invert_yaxis()
    if title:
        plt.title(title)
    plt.grid(True)
    plt.show()


## 1. Why do we need homographies?

A panorama cannot usually be created by only translating one image over another. Translation-only alignment fails when the camera rotates, when the scene is viewed under perspective, or when a planar surface is seen from different viewpoints.

A **homography** is a projective transformation that maps points from one image plane to another image plane.

A homography is often appropriate when:

- The scene is planar, such as a document, wall, floor, poster, painting, or road plane.
- The scene is very far away, so relative depth variation is small.
- The camera undergoes pure rotation around its optical center.

A homography maps homogeneous image coordinates as:

$$
\tilde{\mathbf{x}}' \sim \mathbf{H}\tilde{\mathbf{x}}
$$

where:

$$
\tilde{\mathbf{x}} = \begin{bmatrix}x \\ y \\ 1\end{bmatrix},
\quad
\mathbf{H} \in \mathbb{R}^{3 \times 3}
$$

The symbol $\sim$ means equality up to a non-zero scale. Therefore, $\mathbf{H}$ and $\lambda \mathbf{H}$ represent the same homography for any non-zero scalar $\lambda$.

Although $\mathbf{H}$ has 9 matrix entries, one global scale is arbitrary, so a homography has **8 degrees of freedom**.

### Task 1: Decide whether a homography is a good model

For each case below, write `yes`, `approximately`, or `no`.

1. Two photos of the same flat poster from different viewpoints.
2. Two photos of a nearby 3D object while the camera translates sideways.
3. Two photos of a distant mountain range while the camera rotates.
4. Two photos of a room with strong foreground and background depth differences.
5. Two photos taken from the same position with only camera rotation.


### Code example: create a synthetic image plane

We will use a synthetic grid image as a simple planar object. Later, we will warp this plane using homographies.


In [ ]:
def make_grid_image(width=420, height=300, step=30):
    """Create a simple synthetic planar image with grid lines and markers."""
    img = np.ones((height, width, 3), dtype=np.uint8) * 245

    for x in range(0, width, step):
        cv2.line(img, (x, 0), (x, height - 1), (190, 190, 190), 1)
    for y in range(0, height, step):
        cv2.line(img, (0, y), (width - 1, y), (190, 190, 190), 1)

    cv2.rectangle(img, (60, 50), (width - 60, height - 50), (40, 40, 40), 3)
    cv2.circle(img, (110, 90), 18, (0, 0, 220), -1)
    cv2.circle(img, (width - 110, 90), 18, (0, 160, 0), -1)
    cv2.circle(img, (110, height - 90), 18, (220, 0, 0), -1)
    cv2.putText(img, "PLANAR SURFACE", (88, height // 2), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (20, 20, 20), 2)
    return img

plane_img = make_grid_image()
show_image(plane_img, "Synthetic planar image")


## 2. Applying a homography to points

To apply a homography, convert a 2D point to homogeneous coordinates, multiply by the homography matrix, and then convert back to 2D coordinates.

Given a point $(x, y)$:

$$
\tilde{\mathbf{x}} = \begin{bmatrix}x \\ y \\ 1\end{bmatrix}
$$

Apply the homography:

$$
\begin{bmatrix}u \\ v \\ w\end{bmatrix}
=
\mathbf{H}
\begin{bmatrix}x \\ y \\ 1\end{bmatrix}
$$

Convert back to image coordinates:

$$
\mathbf{x}' = \begin{bmatrix}u / w \\ v / w\end{bmatrix}
$$

This division by $w$ is called **dehomogenization**.


### Code example: transform points with a homography

The function below applies a homography to many points at once.


In [ ]:
def apply_homography(points, H):
    """Apply a 3 by 3 homography to a set of 2D points.

    Parameters
    ----------
    points : array with shape (N, 2)
        Input 2D points.
    H : array with shape (3, 3)
        Homography matrix.

    Returns
    -------
    warped : array with shape (N, 2)
        Transformed 2D points after dehomogenization.
    """
    points = np.asarray(points, dtype=float)
    ones = np.ones((points.shape[0], 1))
    points_h = np.hstack([points, ones])

    warped_h = (H @ points_h.T).T
    warped = warped_h[:, :2] / warped_h[:, 2:3]
    return warped

# Four corners of the synthetic image.
h, w = plane_img.shape[:2]
src_corners = np.array([
    [0, 0],
    [w - 1, 0],
    [w - 1, h - 1],
    [0, h - 1],
], dtype=float)

# A sample projective transformation.
H_demo = np.array([
    [1.00, 0.20, 40.0],
    [0.05, 1.05, 30.0],
    [0.0008, 0.0012, 1.0],
])

warped_corners = apply_homography(src_corners, H_demo)

plot_points(src_corners, labels=["A", "B", "C", "D"], title="Original corners")
plot_points(warped_corners, labels=["A'", "B'", "C'", "D'"], title="Corners after homography")

print("Original corners:\n", src_corners)
print("Warped corners:\n", np.round(warped_corners, 2))


### Task 2: Predict before running

Change one value in `H_demo` and predict the effect before running the cell again.

Try these changes one at a time:

1. Increase `H_demo[0, 2]`. What type of motion does this create?
2. Increase `H_demo[0, 1]`. Does the shape become more slanted?
3. Increase `H_demo[2, 0]`. How does the perspective effect change?
4. Multiply every entry of `H_demo` by 3. Does the output change? Why or why not?


## 3. Image warping with a homography

Point transformation tells us where points move. Image warping tells us how to create a full output image.

A common implementation uses **inverse warping**:

1. For each pixel in the target image, find the corresponding point in the source image.
2. Sample the source image at that point.
3. Write the sampled color into the target pixel.

If $\mathbf{H}$ maps source points to target points, then inverse warping uses:

$$
\tilde{\mathbf{x}}_s \sim \mathbf{H}^{-1}\tilde{\mathbf{x}}_t
$$

Inverse warping avoids holes that may appear with forward warping.


### Code example: warp the synthetic image

OpenCV's `warpPerspective` applies the homography to create a new view. Internally, it uses inverse mapping and interpolation.


In [ ]:
# Define target corners that create a perspective view.
dst_corners = np.array([
    [70, 40],
    [370, 15],
    [395, 260],
    [35, 285],
], dtype=np.float32)

# OpenCV computes the homography from four point pairs.
H_to_view = cv2.getPerspectiveTransform(src_corners.astype(np.float32), dst_corners)

warped_view = cv2.warpPerspective(plane_img, H_to_view, (w, h))

show_image(plane_img, "Source planar image")
show_image(warped_view, "Synthetic perspective view")

print("H_to_view:\n", np.round(H_to_view, 4))


### Task 3: Change the viewpoint

Modify `dst_corners` and rerun the previous code cell.

Questions to answer:

1. Which corner movement creates the strongest perspective distortion?
2. What happens if the corner order is not consistent?
3. Why do black pixels appear near the image boundary?


## 4. Estimating a homography with DLT

In practice, we usually do not know $\mathbf{H}$ directly. Instead, we are given point correspondences:

$$
(x_i, y_i) \leftrightarrow (u_i, v_i)
$$

The homography equation is:

$$
\begin{bmatrix}u_i \\ v_i \\ 1\end{bmatrix}
\sim
\begin{bmatrix}
h_{11} & h_{12} & h_{13} \\
h_{21} & h_{22} & h_{23} \\
h_{31} & h_{32} & h_{33}
\end{bmatrix}
\begin{bmatrix}x_i \\ y_i \\ 1\end{bmatrix}
$$

After multiplying and dehomogenizing:

$$
u_i = \frac{h_{11}x_i + h_{12}y_i + h_{13}}{h_{31}x_i + h_{32}y_i + h_{33}}
$$

$$
v_i = \frac{h_{21}x_i + h_{22}y_i + h_{23}}{h_{31}x_i + h_{32}y_i + h_{33}}
$$

Rearranging gives two linear equations per correspondence:

$$
\begin{bmatrix}
-x_i & -y_i & -1 & 0 & 0 & 0 & u_i x_i & u_i y_i & u_i \\
0 & 0 & 0 & -x_i & -y_i & -1 & v_i x_i & v_i y_i & v_i
\end{bmatrix}
\mathbf{h}
=
\mathbf{0}
$$

where:

$$
\mathbf{h} = \begin{bmatrix}h_{11} & h_{12} & h_{13} & h_{21} & h_{22} & h_{23} & h_{31} & h_{32} & h_{33}\end{bmatrix}^T
$$

Stacking all correspondences gives a homogeneous linear system:

$$
\mathbf{A}\mathbf{h} = \mathbf{0}
$$

The DLT algorithm solves this using SVD. The solution is the right singular vector corresponding to the smallest singular value of $\mathbf{A}$.

Since each correspondence gives two equations and the homography has 8 degrees of freedom, we need at least **4 point correspondences**.


### Code example: implement DLT from scratch

This implementation estimates $\mathbf{H}$ from corresponding points.


In [ ]:
def build_dlt_matrix(src_pts, dst_pts):
    """Build the DLT matrix A for homography estimation."""
    src_pts = np.asarray(src_pts, dtype=float)
    dst_pts = np.asarray(dst_pts, dtype=float)

    if src_pts.shape != dst_pts.shape or src_pts.shape[1] != 2:
        raise ValueError("src_pts and dst_pts must both have shape (N, 2).")

    A = []
    for (x, y), (u, v) in zip(src_pts, dst_pts):
        A.append([-x, -y, -1, 0, 0, 0, u * x, u * y, u])
        A.append([0, 0, 0, -x, -y, -1, v * x, v * y, v])
    return np.asarray(A, dtype=float)


def homography_dlt(src_pts, dst_pts):
    """Estimate a homography using the Direct Linear Transform."""
    A = build_dlt_matrix(src_pts, dst_pts)
    _, _, Vt = np.linalg.svd(A)
    h = Vt[-1]
    H = h.reshape(3, 3)

    # Normalize the matrix so that H[2, 2] is 1 when possible.
    if abs(H[2, 2]) > 1e-12:
        H = H / H[2, 2]
    return H

H_dlt = homography_dlt(src_corners, dst_corners)

print("H from OpenCV:\n", np.round(H_to_view, 4))
print("H from our DLT:\n", np.round(H_dlt, 4))
print("Maximum absolute difference:", np.max(np.abs(H_to_view / H_to_view[2, 2] - H_dlt)))


### Task 4: Four points are enough, but not always safe

Use the code above to answer these questions:

1. Why are four correspondences enough in theory?
2. What happens if three of the four source points are nearly collinear?
3. What happens if you add small noise to the destination points?
4. Why might using more than four correspondences be better in real images?


In [ ]:
# Try a noisy version of the destination corners.
noise_std = 2.0
noisy_dst_corners = dst_corners + np.random.normal(0, noise_std, dst_corners.shape)
H_noisy_4 = homography_dlt(src_corners, noisy_dst_corners)

predicted = apply_homography(src_corners, H_noisy_4)
corner_errors = np.linalg.norm(predicted - dst_corners, axis=1)

print("Noisy destination corners:\n", np.round(noisy_dst_corners, 2))
print("Reprojected corners:\n", np.round(predicted, 2))
print("Corner errors compared with true corners:", np.round(corner_errors, 3))


## 5. Reprojection error and normalized DLT

The DLT system minimizes an algebraic error, not the actual pixel error. To evaluate a homography in image space, use **reprojection error**.

For one correspondence:

$$
e_i = \left\|\mathbf{x}'_i - \pi\left(\mathbf{H}\tilde{\mathbf{x}}_i\right)\right\|_2
$$

where $\pi$ dehomogenizes a homogeneous point.

DLT can also be numerically unstable when image coordinates are large. A standard improvement is **normalized DLT**:

1. Shift points so their centroid is at the origin.
2. Scale points so their average distance from the origin is $\sqrt{2}$.
3. Estimate the homography in normalized coordinates.
4. Denormalize the result.

If $\mathbf{T}_s$ normalizes source points and $\mathbf{T}_d$ normalizes destination points, then:

$$
\mathbf{H} = \mathbf{T}_d^{-1}\mathbf{H}_{norm}\mathbf{T}_s
$$


### Code example: normalized DLT

The following code compares standard DLT and normalized DLT on noisy correspondences.


In [ ]:
def normalize_points(points):
    """Normalize 2D points to zero centroid and average distance sqrt(2)."""
    points = np.asarray(points, dtype=float)
    centroid = points.mean(axis=0)
    centered = points - centroid
    distances = np.linalg.norm(centered, axis=1)
    mean_distance = distances.mean()

    if mean_distance < 1e-12:
        raise ValueError("Cannot normalize points with near-zero spread.")

    scale = math.sqrt(2) / mean_distance
    T = np.array([
        [scale, 0, -scale * centroid[0]],
        [0, scale, -scale * centroid[1]],
        [0, 0, 1],
    ])

    normalized = apply_homography(points, T)
    return normalized, T


def homography_dlt_normalized(src_pts, dst_pts):
    """Estimate a homography using normalized DLT."""
    src_norm, T_src = normalize_points(src_pts)
    dst_norm, T_dst = normalize_points(dst_pts)

    H_norm = homography_dlt(src_norm, dst_norm)
    H = np.linalg.inv(T_dst) @ H_norm @ T_src

    if abs(H[2, 2]) > 1e-12:
        H = H / H[2, 2]
    return H


def reprojection_errors(src_pts, dst_pts, H):
    """Compute Euclidean reprojection errors in pixels."""
    projected = apply_homography(src_pts, H)
    return np.linalg.norm(projected - dst_pts, axis=1)

# Generate many random points on the source image.
num_points = 60
src_random = np.column_stack([
    np.random.uniform(20, w - 20, num_points),
    np.random.uniform(20, h - 20, num_points),
])

dst_clean = apply_homography(src_random, H_to_view)
dst_noisy = dst_clean + np.random.normal(0, 1.5, dst_clean.shape)

H_plain = homography_dlt(src_random, dst_noisy)
H_norm = homography_dlt_normalized(src_random, dst_noisy)

errors_plain = reprojection_errors(src_random, dst_clean, H_plain)
errors_norm = reprojection_errors(src_random, dst_clean, H_norm)

print("Mean reprojection error with plain DLT:", round(errors_plain.mean(), 4))
print("Mean reprojection error with normalized DLT:", round(errors_norm.mean(), 4))
print("Median reprojection error with plain DLT:", round(np.median(errors_plain), 4))
print("Median reprojection error with normalized DLT:", round(np.median(errors_norm), 4))


In [ ]:
plt.figure(figsize=(7, 5))
plt.hist(errors_plain, bins=15, alpha=0.6, label="Plain DLT")
plt.hist(errors_norm, bins=15, alpha=0.6, label="Normalized DLT")
plt.xlabel("Reprojection error in pixels")
plt.ylabel("Count")
plt.title("Reprojection error comparison")
plt.legend()
plt.show()


### Task 5: Study noise and stability

Change the value of the noise standard deviation in the previous code.

Suggested values: `0.5`, `1.5`, `4.0`, and `8.0`.

Questions to answer:

1. Does normalized DLT always win?
2. When the noise is large, what happens to the reprojection error distribution?
3. Why is the median sometimes more informative than the mean?
4. What would happen if all correspondences were clustered in a tiny region of the image?


## 6. Robust homography estimation with RANSAC

Real feature matching usually contains bad correspondences. A single bad match can corrupt a least-squares homography estimate.

RANSAC, or Random Sample Consensus, handles outliers by repeatedly:

1. Sampling the minimum number of correspondences needed to fit the model.
2. Estimating a candidate model from that sample.
3. Counting how many correspondences agree with the model.
4. Keeping the model with the largest inlier set.
5. Re-estimating the model using all inliers.

For homography estimation, the minimum sample size is:

$$
s = 4
$$

If the inlier ratio is $w$ and the desired success probability is $p$, a common estimate for the number of RANSAC iterations is:

$$
N = \frac{\log(1 - p)}{\log(1 - w^s)}
$$

When the outlier ratio is $e$, the inlier ratio is:

$$
w = 1 - e
$$


### Code example: estimate a homography with RANSAC

We will simulate good matches and bad matches. Then we will use RANSAC to recover the homography.


In [ ]:
def required_ransac_iterations(inlier_ratio, sample_size=4, success_probability=0.99):
    """Compute the standard RANSAC iteration estimate."""
    inlier_ratio = np.clip(inlier_ratio, 1e-9, 1 - 1e-9)
    numerator = math.log(1 - success_probability)
    denominator = math.log(1 - inlier_ratio ** sample_size)
    return math.ceil(numerator / denominator)

for outlier_ratio in [0.1, 0.25, 0.4, 0.5]:
    inlier_ratio = 1 - outlier_ratio
    N = required_ransac_iterations(inlier_ratio)
    print(f"Outlier ratio {outlier_ratio:.0%}: about {N} iterations for 99% success")


In [ ]:
def ransac_homography(src_pts, dst_pts, threshold=4.0, max_iterations=1000, seed=0):
    """Estimate a homography with RANSAC.

    Parameters
    ----------
    src_pts : array with shape (N, 2)
        Source points.
    dst_pts : array with shape (N, 2)
        Destination points.
    threshold : float
        Maximum reprojection error for a correspondence to count as an inlier.
    max_iterations : int
        Number of random samples.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    best_H : array with shape (3, 3)
        Estimated homography after refitting with all inliers.
    best_inliers : boolean array with shape (N,)
        Inlier mask.
    """
    rng = np.random.default_rng(seed)
    src_pts = np.asarray(src_pts, dtype=float)
    dst_pts = np.asarray(dst_pts, dtype=float)

    if len(src_pts) < 4:
        raise ValueError("At least four correspondences are required.")

    best_H = None
    best_inliers = None
    best_count = -1
    n = len(src_pts)

    for _ in range(max_iterations):
        sample_idx = rng.choice(n, size=4, replace=False)

        try:
            H_candidate = homography_dlt_normalized(src_pts[sample_idx], dst_pts[sample_idx])
        except np.linalg.LinAlgError:
            continue
        except ValueError:
            continue

        errors = reprojection_errors(src_pts, dst_pts, H_candidate)
        inliers = errors < threshold
        count = int(inliers.sum())

        if count > best_count:
            best_count = count
            best_inliers = inliers
            best_H = H_candidate

    if best_inliers is None or best_inliers.sum() < 4:
        raise RuntimeError("RANSAC failed to find a valid homography.")

    # Refit using all inliers.
    best_H = homography_dlt_normalized(src_pts[best_inliers], dst_pts[best_inliers])
    return best_H, best_inliers

# Simulate correspondences.
num_inliers = 80
num_outliers = 35
noise_std = 1.8

src_inliers = np.column_stack([
    np.random.uniform(20, w - 20, num_inliers),
    np.random.uniform(20, h - 20, num_inliers),
])
dst_inliers = apply_homography(src_inliers, H_to_view)
dst_inliers += np.random.normal(0, noise_std, dst_inliers.shape)

src_outliers = np.column_stack([
    np.random.uniform(0, w, num_outliers),
    np.random.uniform(0, h, num_outliers),
])
dst_outliers = np.column_stack([
    np.random.uniform(0, w, num_outliers),
    np.random.uniform(0, h, num_outliers),
])

src_matches = np.vstack([src_inliers, src_outliers])
dst_matches = np.vstack([dst_inliers, dst_outliers])

H_ransac, inliers = ransac_homography(
    src_matches,
    dst_matches,
    threshold=5.0,
    max_iterations=800,
    seed=3,
)

print("Number of matches:", len(src_matches))
print("Number of RANSAC inliers:", int(inliers.sum()))
print("Estimated inlier ratio:", round(float(inliers.mean()), 3))
print("Mean inlier reprojection error:", round(reprojection_errors(src_matches[inliers], dst_matches[inliers], H_ransac).mean(), 3))


In [ ]:
def plot_matches(src_pts, dst_pts, inlier_mask, title):
    """Plot source and destination point pairs in a single coordinate system."""
    plt.figure(figsize=(8, 6))
    for s, d, is_inlier in zip(src_pts, dst_pts, inlier_mask):
        if is_inlier:
            plt.plot([s[0], d[0]], [s[1], d[1]], linewidth=0.6, alpha=0.6)
        else:
            plt.plot([s[0], d[0]], [s[1], d[1]], linewidth=0.6, alpha=0.2, linestyle="--")
    plt.scatter(src_pts[:, 0], src_pts[:, 1], s=12, label="source")
    plt.scatter(dst_pts[:, 0], dst_pts[:, 1], s=12, label="destination")
    plt.gca().invert_yaxis()
    plt.axis("equal")
    plt.grid(True)
    plt.legend()
    plt.title(title)
    plt.show()

plot_matches(src_matches, dst_matches, inliers, "RANSAC inliers and outliers")


### Task 6: Tune RANSAC

Change `threshold`, `max_iterations`, `num_outliers`, and `noise_std`.

Questions to answer:

1. If the threshold is too small, what happens to true matches?
2. If the threshold is too large, what happens to false matches?
3. Why does RANSAC need more iterations when the outlier ratio increases?
4. Why do we re-estimate $\mathbf{H}$ using all inliers at the end?


## 7. Image rectification

Image rectification uses a homography to transform a plane into a more useful view.

Examples include:

- A document captured at an angle.
- A floor pattern seen under perspective.
- A painting or poster on a wall.
- A road plane in a driving scene.

The common workflow is:

1. Select four corners of a planar region in the input image.
2. Choose four corners of a desired rectangular output image.
3. Estimate the homography from input corners to output corners.
4. Warp the input image into the rectified coordinate system.


### Code example: rectify a synthetic document

We will create a document, view it under perspective, then rectify it back to a front-facing view.


In [ ]:
def make_document(width=360, height=240):
    """Create a synthetic document-like image."""
    doc = np.ones((height, width, 3), dtype=np.uint8) * 255
    cv2.rectangle(doc, (0, 0), (width - 1, height - 1), (30, 30, 30), 3)
    cv2.putText(doc, "HOMOGRAPHY", (35, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)
    cv2.putText(doc, "DLT + RANSAC", (45, 105), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 0), 2)
    for y_line in [145, 175, 205]:
        cv2.line(doc, (35, y_line), (width - 35, y_line), (80, 80, 80), 2)
    return doc

doc = make_document()
doc_h, doc_w = doc.shape[:2]

doc_src = np.array([
    [0, 0],
    [doc_w - 1, 0],
    [doc_w - 1, doc_h - 1],
    [0, doc_h - 1],
], dtype=np.float32)

doc_view_corners = np.array([
    [90, 30],
    [380, 70],
    [330, 265],
    [45, 215],
], dtype=np.float32)

canvas_w, canvas_h = 460, 320
H_doc_to_view = cv2.getPerspectiveTransform(doc_src, doc_view_corners)
view = cv2.warpPerspective(doc, H_doc_to_view, (canvas_w, canvas_h))

show_image(doc, "Original document")
show_image(view, "Perspective document view")


In [ ]:
# Rectify the document back to a rectangle.
rectified_size = (doc_w, doc_h)
rect_dst = np.array([
    [0, 0],
    [doc_w - 1, 0],
    [doc_w - 1, doc_h - 1],
    [0, doc_h - 1],
], dtype=np.float32)

H_view_to_doc = homography_dlt_normalized(doc_view_corners, rect_dst)
rectified = cv2.warpPerspective(view, H_view_to_doc, rectified_size)

show_image(view, "Input perspective view")
show_image(rectified, "Rectified view")


### Task 7: Rectification analysis

Answer these questions after running the rectification example:

1. Why do we only need four corners for this example?
2. What would happen if one selected corner were inaccurate by 20 pixels?
3. Which visual details are restored by rectification?
4. Which details cannot be recovered once they are lost due to sampling, blur, or occlusion?


## 8. Automatic correspondences with feature matching

Manual corner selection is useful for learning, but many computer vision systems detect and match features automatically.

A typical correspondence pipeline is:

1. Detect feature points, such as corners or blobs.
2. Describe each feature using a descriptor.
3. Match descriptors between images.
4. Use RANSAC to estimate a homography and reject bad matches.

In this section, we use ORB features because they are available in OpenCV and run quickly.

Feature matching can fail when:

- The image has too little texture.
- The image has repeated patterns.
- The viewpoint change is too large.
- The image contains moving objects or non-planar structure.


### Code example: ORB matching plus homography RANSAC

We will match features between the original synthetic plane and its perspective view.


In [ ]:
# Convert images to grayscale for feature detection.
gray_src = cv2.cvtColor(plane_img, cv2.COLOR_BGR2GRAY)
gray_dst = cv2.cvtColor(warped_view, cv2.COLOR_BGR2GRAY)

orb = cv2.ORB_create(nfeatures=800)
kp1, des1 = orb.detectAndCompute(gray_src, None)
kp2, des2 = orb.detectAndCompute(gray_dst, None)

print("Number of keypoints in source:", len(kp1))
print("Number of keypoints in target:", len(kp2))

if des1 is None or des2 is None:
    raise RuntimeError("ORB did not find enough descriptors. Try adding more texture to the image.")

matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = matcher.match(des1, des2)
matches = sorted(matches, key=lambda m: m.distance)

# Use the best matches for homography estimation.
num_best = min(80, len(matches))
good_matches = matches[:num_best]

src_orb = np.float32([kp1[m.queryIdx].pt for m in good_matches])
dst_orb = np.float32([kp2[m.trainIdx].pt for m in good_matches])

H_cv, mask_cv = cv2.findHomography(src_orb, dst_orb, cv2.RANSAC, 5.0)
print("Number of matches used:", len(good_matches))
print("Number of OpenCV RANSAC inliers:", int(mask_cv.sum()) if mask_cv is not None else 0)


In [ ]:
match_vis = cv2.drawMatches(
    plane_img,
    kp1,
    warped_view,
    kp2,
    good_matches[:30],
    None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)

show_image(match_vis, "Top ORB matches", size=(12, 6))

if H_cv is not None:
    aligned = cv2.warpPerspective(plane_img, H_cv, (w, h))
    show_image(warped_view, "Target perspective view")
    show_image(aligned, "Source warped using feature-estimated homography")


### Task 8: Diagnose feature matching

Try modifying the synthetic image in `make_grid_image`.

Experiments:

1. Remove the colored circles. Does matching become harder?
2. Add more repeated grid lines. Does matching become more ambiguous?
3. Add unique text or symbols. Does matching become easier?
4. Increase the viewpoint distortion. When does matching begin to fail?

Write down one reason why feature matching and RANSAC should be used together.


## 9. Mini-project: build a simple panorama-style alignment demo

A real panorama system has many parts, but this mini-project focuses on the homography part.

### Goal

Create two overlapping views of the same synthetic planar image, estimate the homography between them, and place one image into the coordinate system of the other.

### Suggested steps

1. Create a larger synthetic image with strong visual features.
2. Generate two overlapping perspective views using two different homographies.
3. Detect and match ORB features between the two views.
4. Estimate a homography with RANSAC.
5. Warp one view into the coordinate system of the other.
6. Create a simple blended canvas.

### Reflection questions

1. What parts of the result look correct?
2. Where do you see misalignment?
3. Would translation-only alignment solve the same case?
4. What assumptions does this synthetic example satisfy that real panoramas may violate?


### Code scaffold: panorama-style alignment

This scaffold is intentionally short. Use the previous sections as references.


In [ ]:
# Step 1: create a base planar image.
base = make_grid_image(width=620, height=360, step=35)
base_h, base_w = base.shape[:2]
base_corners = np.array([
    [0, 0],
    [base_w - 1, 0],
    [base_w - 1, base_h - 1],
    [0, base_h - 1],
], dtype=np.float32)

# Step 2: generate two different views of the same plane.
view_size = (520, 320)
view1_corners = np.array([
    [20, 25],
    [500, 10],
    [485, 300],
    [35, 310],
], dtype=np.float32)
view2_corners = np.array([
    [-60, 35],
    [430, 20],
    [505, 300],
    [10, 315],
], dtype=np.float32)

H_base_to_1 = cv2.getPerspectiveTransform(base_corners, view1_corners)
H_base_to_2 = cv2.getPerspectiveTransform(base_corners, view2_corners)

view1 = cv2.warpPerspective(base, H_base_to_1, view_size)
view2 = cv2.warpPerspective(base, H_base_to_2, view_size)

show_image(view1, "View 1")
show_image(view2, "View 2")


In [ ]:
# Step 3: match ORB features between the two views.
g1 = cv2.cvtColor(view1, cv2.COLOR_BGR2GRAY)
g2 = cv2.cvtColor(view2, cv2.COLOR_BGR2GRAY)

kp1, des1 = orb.detectAndCompute(g1, None)
kp2, des2 = orb.detectAndCompute(g2, None)

if des1 is None or des2 is None:
    raise RuntimeError("Not enough descriptors found for panorama-style alignment.")

matches = matcher.match(des1, des2)
matches = sorted(matches, key=lambda m: m.distance)
matches = matches[:120]

pts1 = np.float32([kp1[m.queryIdx].pt for m in matches])
pts2 = np.float32([kp2[m.trainIdx].pt for m in matches])

H_1_to_2, mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, 5.0)
print("Matches:", len(matches))
print("RANSAC inliers:", int(mask.sum()) if mask is not None else 0)

match_vis = cv2.drawMatches(
    view1, kp1, view2, kp2, matches[:40], None,
    flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS,
)
show_image(match_vis, "Matches between two synthetic views", size=(12, 5))


In [ ]:
# Step 4: warp view 1 into the coordinate system of view 2.
if H_1_to_2 is None:
    raise RuntimeError("Homography estimation failed.")

canvas_w, canvas_h = 720, 420
translation = np.array([
    [1, 0, 100],
    [0, 1, 50],
    [0, 0, 1],
], dtype=float)

view1_on_canvas = cv2.warpPerspective(view1, translation @ H_1_to_2, (canvas_w, canvas_h))
view2_on_canvas = cv2.warpPerspective(view2, translation, (canvas_w, canvas_h))

# Simple overlay: keep view2 where it has pixels, otherwise use warped view1.
mask2 = np.any(view2_on_canvas > 0, axis=2)
blend = view1_on_canvas.copy()
blend[mask2] = view2_on_canvas[mask2]

show_image(view1_on_canvas, "View 1 warped to canvas")
show_image(view2_on_canvas, "View 2 on canvas")
show_image(blend, "Simple panorama-style overlay")


## 10. Self-check summary

You should now be able to answer the following without looking back:

1. What does the scale ambiguity of a homography mean?
2. Why does a homography have 8 degrees of freedom?
3. Why are at least 4 point correspondences required?
4. What is the difference between algebraic error and reprojection error?
5. Why does normalized DLT improve numerical stability?
6. What role does the RANSAC threshold play?
7. Why is a homography useful for image rectification?
8. Why is homography-based stitching limited when a scene has large depth variation and camera translation?

## Extension tasks

- Implement bilinear interpolation for inverse warping from scratch.
- Replace ORB with another feature detector and compare results.
- Add nonlinear refinement that minimizes reprojection error after DLT.
- Use two real images of a poster or book cover and estimate a homography.
- Build an interactive corner picker for rectification.
